In [29]:
! nvidia-smi

Tue Mar 31 09:10:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:81:00.0 Off |                  Off |
| 64%   34C    P8             18W /  450W |   30901MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Install required packages

In [1]:
! pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126 -q

In [2]:
! pip install transformers==4.56.1 sacremoses evaluate sacrebleu beautifulsoup4 sentencepiece -q

# Imports

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer, MarianMTModel, MarianTokenizer, pipeline
import os
import torch
import evaluate
from collections import Counter
import gc
# from kaggle_secrets import UserSecretsClient

# Load data

## Flores+
Covered languages: Nigerian Fulfulde (fuv).

In [4]:
# Kaggle Secrets Client to manage API keys securely in Kaggle notebooks
# user_secrets = UserSecretsClient()

# # Set your HF token as an environment variable
# os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [5]:
fuv_flores_test = load_dataset("openlanguagedata/flores_plus", "fuv_Latn", split="devtest").to_pandas()
eng_flores_test = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest").to_pandas()

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

In [6]:
# Join the two dataframes on the 'id' column
flores_df = pd.merge(fuv_flores_test, eng_flores_test, on='id', suffixes=('_fuv', '_eng'))

# Only keep the 'text_eng' and 'text_fuv' columns
flores_df = flores_df[['text_eng', 'text_fuv']]

# Rename the columns to 'eng_Latn' and 'fuv_Latn'
flores_df.rename(columns={'text_eng': 'eng_Latn', 'text_fuv': 'fuv_Latn'}, inplace=True)

flores_df

,eng_Latn,fuv_Latn
0,"""We now have 4-month-old mice that are non-dia...",“Jotta minɗo mari ɓera je wala diyabetis bawo ...
1,"Dr. Ehud Ur, professor of medicine at Dalhousi...","Dr. Ehud Ur, mallumjo manga ha jangirde lekki ..."
2,"Like some other experts, he is skeptical about...","Bana ɓe arti wawi sossay, o wala tabbaci ko ny..."
3,"On Monday, Sara Danius, permanent secretary of...","Nyande Altin, Sra Deniyus. Kanko on mo hakkili..."
4,"Danius said, ""Right now we are doing nothing. ...","Danius wi “jotta wala ko min waɗata, mi fiyi w..."
...,...,...
1007,"As the areas are sparsely populated, and light...",Baabe may ɗon yaaji wala yimɓe wonay jur ha to...
1008,Japanese work culture is more hierarchical and...,Japanese ɗengal ɗon yaha ta'irde be ta'irde e ...
1009,"Suits are standard business attire, and cowork...",Tokgore kam jun on kare hani goɗɗo wata ha pel...
1010,"Workplace harmony is crucial, emphasizing grou...","Jonde jam ha baabal kugal ɗo mari nafu masin, ..."


## BOUQuET
Covered languages: Nigerian Fulfulde (fuv), Pulaar (fuc).

In [7]:
fuc_bouquet_test = load_dataset("facebook/bouquet", "fuc_Latn", split="test").to_pandas()
fuv_bouquet_test = load_dataset("facebook/bouquet", "fuv_Latn", split="test").to_pandas()

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/550 [00:00<?, ?it/s]

In [8]:
# Rename "src_text"
fuc_bouquet_test.rename(columns={'src_text': 'fuc_Latn', 'tgt_text': 'eng_Latn'}, inplace=True)
fuv_bouquet_test.rename(columns={'src_text': 'fuv_Latn'}, inplace=True)

# Merge the two dataframes on the "uniq_id" column
bouquet_df = pd.merge(fuc_bouquet_test, fuv_bouquet_test, on='uniq_id')

# Only keep the "fuc_Latn", "fuv_Latn", and "eng_Latn" columns
bouquet_df = bouquet_df[['fuc_Latn', 'fuv_Latn', 'eng_Latn']]

bouquet_df

,fuc_Latn,fuv_Latn,eng_Latn
0,Aɗe waawi waɗde maakoroni e maafe bekamel « pa...,"""Aɗa waawi waɗde paasta e nofru """"Pastitsio"""" ...","You can easily make pasta ""Pastitsio"" with bec..."
1,"Ko kaajeteɗaa ko heen fof ko ɗum maakoroni, te...","Ko njiɗɗaa fof ko paasta, na’i dowri, njuumri,...","All you will need is pasta, ground beef, onion..."
2,"Njuɗaa teewu ngu wondude e basalle, pawa ɗum b...","Sauuté 6uu6ol les ngol e tineeje cfee, 6uucca ...",Sauté the ground beef with the onions and put ...
3,"Pasna maakoroni o, ngittaamo e jeyngol.","Buuɓnu paasta oo, itta ɗum e nguleeki.",Boil the pasta and remove it from the heat.
4,"Pewna soos bekamel no leeliri, so ɓuuɓi tan, m...","Mbaɗaa beechamel seese-seese, ɓeydu egg oo so ...",Make the bechamel slowly and add the egg when ...
...,...,...,...
849,"Madam Hooreejo Suudu Sarɗiiji, Banndiraaɓe Rew...","Madam Wolwoowo Saare, rowɓe be worɓe Laatiɓe n...","Madam House Speaker, ladies and gentlemen Memb..."
850,Miɗo ɗoo ɗaɓɓude woote hoolaare e ndeeɗoo suud...,Mi daro ha yeeso Saare ɗo ngam ɗaɓɓutuki yerdu...,I am appearing before this House to request yo...
851,"Njeñtudi woote ɗee ina laaɓti, ina hollita bal...",Ngasnaaɗi cuɓol mai laatake ɗuɗɗum masin nden ...,The election results have been overwhelming an...
852,Woote ɗe ɓiɗɓe leydi ndii kollitii e dow welli...,Cuɓol jei ha himɓe lesdi heɓi holluki be hoore...,An election in which citizens have freely and ...


## UDHR
Covered languages: Dagbani (dag).

In [9]:
# Load the UDHR (Universal Declaration of Human Rights) dataset for Dagbani (dag)
udhr_dag_url = "http://efele.net/udhr/d/udhr_dag.xml"
udhr_eng_url = "http://efele.net/udhr/d/udhr_eng.xml"

def load_udhr(url):
    if os.path.exists(url.split("/")[-1]):
        with open(url.split("/")[-1], "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "xml")
    else:
        response = requests.get(url)
        # Download the file and save it locally for future use
        with open(url.split("/")[-1], "w", encoding="utf-8") as f:
            f.write(response.text)
        soup = BeautifulSoup(response.content, "xml")
    text = [p.get_text() for p in soup.find_all("para")]
    return text

In [10]:
udhr_eng = load_udhr(udhr_eng_url)

# Merge English paragraphs 1 through 7
udhr_eng_para_1 = " ".join(udhr_eng[:7])

# Merge English paragraphs 8 through 10
udhr_eng_para_2 = " ".join(udhr_eng[7:10])

udhr_eng = [udhr_eng_para_1, udhr_eng_para_2] + udhr_eng[10:]

In [11]:
udhr_dag = load_udhr(udhr_dag_url)

# Merge Dagbani paragraphs 1 through 3
udhr_dag_para_1 = " ".join(udhr_dag[:3])

# Merge Dagbani paragraphs 4 through 8
udhr_dag_para_2 = " ".join(udhr_dag[3:8])

udhr_dag = [udhr_dag_para_1, udhr_dag_para_2] + udhr_dag[8:]

In [12]:
assert len(udhr_dag) == len(udhr_eng), f"The number of paragraphs in Dagbani and English UDHR should be the same. Found {len(udhr_dag)} and {len(udhr_eng)} respectively."

In [13]:
# Convert to Pandas DataFrame
udhr_df = pd.DataFrame({
    "eng_Latn": udhr_eng,
    "dag_Latn": udhr_dag
})

udhr_df.head(5)

,eng_Latn,dag_Latn
0,Whereas recognition of the inherent dignity an...,"Yɛlimaŋli, bɛhigu kam yi yɛn mali kɔre ni dari..."
1,"Now, therefore, The General Assembly Proclaims...","Dinzugu, pumpɔŋɔ, Jɛneral Asɛmbli Laɣiŋgu ni d..."
2,All human beings are born free and equal in di...,"Sal' la sala. Bɛhig' be sokam sanimi, din pa l..."
3,Everyone is entitled to all the rights and fre...,Zaligu dinkam be litaafi ŋɔ ni nyɛla din maani...
4,"Furthermore, no distinction shall be made on t...",Din nyaaŋa di ka soli ka so ʒem o kpee ni o ti...


# Preprocess Data

In [14]:
# this code is adapted from  the Stopes repo of the NLLB team
# https://github.com/facebookresearch/stopes/blob/main/stopes/pipelines/monolingual/monolingual_line_processor.py#L214

import re
import sys
import typing as tp
import unicodedata
from sacremoses import MosesPunctNormalizer


mpn = MosesPunctNormalizer(lang="en")
mpn.substitutions = [
    (re.compile(r), sub) for r, sub in mpn.substitutions
]


def get_non_printing_char_replacer(replace_by: str = " ") -> tp.Callable[[str], str]:
    non_printable_map = {
        ord(c): replace_by
        for c in (chr(i) for i in range(sys.maxunicode + 1))
        # same as \p{C} in perl
        # see https://www.unicode.org/reports/tr44/#General_Category_Values
        if unicodedata.category(c) in {"C", "Cc", "Cf", "Cs", "Co", "Cn"}
    }

    def replace_non_printing_char(line) -> str:
        return line.translate(non_printable_map)

    return replace_non_printing_char

replace_nonprint = get_non_printing_char_replacer(" ")

def preproc(text):
    clean = mpn.normalize(text)
    clean = replace_nonprint(clean)
    # replace 𝓕𝔯𝔞𝔫𝔠𝔢𝔰𝔠𝔞 by Francesca
    clean = unicodedata.normalize("NFKC", clean)
    return clean

In [15]:
# Get the vocabulary of one dataset to check if the preprocessing is working correctly
print("Before preprocessing:")
count_before = Counter("".join(flores_df['eng_Latn']))
print("Flores English vocab size:", len(count_before))

datasets = {
    'flores': flores_df,
    'bouquet': bouquet_df,
    'udhr': udhr_df,
    # 'pontoon': pontoon_df
}

# Apply preprocessing to all datasets
for name, df in datasets.items():
    print(f"Processing {name} dataset...")
    for col in df.columns:
        df[col] = df[col].apply(preproc)

print("\nAfter preprocessing:")
count_after = Counter("".join(flores_df['eng_Latn']))
print("Flores English vocab size:", len(count_after))

for char in count_before:
    if char not in count_after:
        print(f"Character '{char}' (freq: {count_before[char]}) was removed during preprocessing.")

Before preprocessing:
Flores English vocab size: 106
Processing flores dataset...
Processing bouquet dataset...
Processing udhr dataset...

After preprocessing:
Flores English vocab size: 97
Character '—' (freq: 10) was removed during preprocessing.
Character '–' (freq: 4) was removed during preprocessing.
Character '’' (freq: 14) was removed during preprocessing.
Character '¾' (freq: 1) was removed during preprocessing.
Character '½' (freq: 1) was removed during preprocessing.
Character '“' (freq: 9) was removed during preprocessing.
Character '‘' (freq: 1) was removed during preprocessing.
Character '”' (freq: 9) was removed during preprocessing.
Character '​' (freq: 2) was removed during preprocessing.
Character '²' (freq: 2) was removed during preprocessing.


# Evaluation

In [16]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
ter_metric = evaluate.load("ter")

def metrics_calc(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    bleu_result = bleu_metric.compute(predictions = preds, references = labels)
    spm_result = bleu_metric.compute(predictions = preds, references = labels, tokenize='flores101')
    chrf_result = chrf_metric.compute(predictions = preds, references = labels, word_order=2)
    ter_result = ter_metric.compute(predictions = preds, references = labels)

    result = {
        'bleu': bleu_result['score'],
        'spbleu': spm_result['score'],
        'chrf++': chrf_result['score'],
        'ter': ter_result['score']
    }
    result = {k: round(v, 2) for k, v in result.items()}

    return result

In [17]:
def evaluate_mt_model(model_name, inference_fn, data, **kwargs):
    results = []

    for dataset_name, df in data.items():
        print(f"Evaluating on {dataset_name} dataset...")
        langs = df.columns.tolist()
        langs.pop(langs.index('eng_Latn'))
        eng_texts = df['eng_Latn'].tolist()
        for lang in langs:
            # From English to the target language
            print(f"Translating from English to {lang}...")
            translations = inference_fn(model_name, eng_texts, "eng_Latn", lang, **kwargs)
            if translations is None:
                continue
            references = df[lang].tolist()
            from_metrics = metrics_calc(translations, references)
            from_prefix = f"eng_Latn-{lang}"
            from_metrics = {f"{from_prefix}_{k}": v for k, v in from_metrics.items()}
            print(f"Results for English to {lang}: {from_metrics}")

            # From the target language to English
            print(f"Translating from {lang} to English...")
            translations = inference_fn(model_name, references, lang, "eng_Latn", **kwargs)
            if translations is None:
                continue
            to_metrics = metrics_calc(translations, eng_texts)
            to_prefix = f"{lang}-eng_Latn"
            to_metrics = {f"{to_prefix}_{k}": v for k, v in to_metrics.items()}
            print(f"Results for {lang} to English: {to_metrics}\n")

            results.append({
                'model': model_name,
                'dataset': dataset_name,
                'language_pair': f"eng_Latn-{lang}",
                **from_metrics,
                **to_metrics
            })

    # return results
    return pd.DataFrame(results)

## NLLB-200 3B
Supported languages: Nigerian Fulfulde (fuv).

In [18]:
NLLB_600M_MODEL_ID = "facebook/nllb-200-distilled-600M"
NLLB_3B_MODEL_ID = "facebook/nllb-200-3.3B"

nllb_600m_tokenizer = AutoTokenizer.from_pretrained(NLLB_600M_MODEL_ID)

# Get the list of supported languages for the NLLB models
NLLB_SUPPORTED_LANGS = nllb_600m_tokenizer.special_tokens_map['additional_special_tokens']

In [19]:
def nllb_inference(model_name, src_texts, src_lang, tgt_lang, **kwargs):
    # Check if the model supports the source and target languages
    if src_lang not in NLLB_SUPPORTED_LANGS:
        print(f"Source language '{src_lang}' is not supported by the model.")
        return None
    if tgt_lang not in NLLB_SUPPORTED_LANGS:
        print(f"Target language '{tgt_lang}' is not supported by the model.")
        return None

    # Use pipeline
    translation_pipeline = pipeline(
        "translation",
        model=model_name,
        src_lang=src_lang,
        tgt_lang=tgt_lang,
        device=0 if torch.cuda.is_available() else -1,
        **kwargs
    )

    results = translation_pipeline(src_texts)

    return [res['translation_text'] for res in results]

In [20]:
nllb_3b_results = evaluate_mt_model(NLLB_3B_MODEL_ID, nllb_inference, datasets, batch_size=48)
nllb_3b_results

Evaluating on flores dataset...
Translating from English to fuv_Latn...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


Results for English to fuv_Latn: {'eng_Latn-fuv_Latn_bleu': 3.97, 'eng_Latn-fuv_Latn_spbleu': 5.28, 'eng_Latn-fuv_Latn_chrf++': 22.94, 'eng_Latn-fuv_Latn_ter': 100.88}
Translating from fuv_Latn to English...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

...

Results for English to fuv_Latn: {'eng_Latn-fuv_Latn_bleu': 6.5, 'eng_Latn-fuv_Latn_spbleu': 9.63, 'eng_Latn-fuv_Latn_chrf++': 24.51, 'eng_Latn-fuv_Latn_ter': 94.96}
Translating from fuv_Latn to English...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


Results for fuv_Latn to English: {'fuv_Latn-eng_Latn_bleu': 18.08, 'fuv_Latn-eng_Latn_spbleu': 19.04, 'fuv_Latn-eng_Latn_chrf++': 36.27, 'fuv_Latn-eng_Latn_ter': 78.17}

Evaluating on udhr dataset...
Translating from English to dag_Latn...
Target language 'dag_Latn' is not supported by the model.


,model,dataset,language_pair,eng_Latn-fuv_Latn_bleu,eng_Latn-fuv_Latn_spbleu,eng_Latn-fuv_Latn_chrf++,eng_Latn-fuv_Latn_ter,fuv_Latn-eng_Latn_bleu,fuv_Latn-eng_Latn_spbleu,fuv_Latn-eng_Latn_chrf++,fuv_Latn-eng_Latn_ter
0,facebook/nllb-200-3.3B,flores,eng_Latn-fuv_Latn,3.97,5.28,22.94,100.88,11.55,13.72,32.40,91.63
1,facebook/nllb-200-3.3B,bouquet,eng_Latn-fuv_Latn,6.50,9.63,24.51,94.96,18.08,19.04,36.27,78.17


## NLLB-200 distilled 600M
Supported languages: Nigerian Fulfulde (fuv).

In [21]:
# Free up GPU memory
gc.collect()
torch.cuda.empty_cache()

In [22]:
nllb_600m_results = evaluate_mt_model(NLLB_600M_MODEL_ID, nllb_inference, datasets, batch_size=64)
nllb_600m_results

Evaluating on flores dataset...
Translating from English to fuv_Latn...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device set to use cuda:0


...

Device set to use cuda:0


Results for English to fuv_Latn: {'eng_Latn-fuv_Latn_bleu': 4.85, 'eng_Latn-fuv_Latn_spbleu': 5.81, 'eng_Latn-fuv_Latn_chrf++': 20.88, 'eng_Latn-fuv_Latn_ter': 135.7}
Translating from fuv_Latn to English...


Device set to use cuda:0


Results for fuv_Latn to English: {'fuv_Latn-eng_Latn_bleu': 14.75, 'fuv_Latn-eng_Latn_spbleu': 15.95, 'fuv_Latn-eng_Latn_chrf++': 32.04, 'fuv_Latn-eng_Latn_ter': 82.87}

Evaluating on udhr dataset...
Translating from English to dag_Latn...
Target language 'dag_Latn' is not supported by the model.


,model,dataset,language_pair,eng_Latn-fuv_Latn_bleu,eng_Latn-fuv_Latn_spbleu,eng_Latn-fuv_Latn_chrf++,eng_Latn-fuv_Latn_ter,fuv_Latn-eng_Latn_bleu,fuv_Latn-eng_Latn_spbleu,fuv_Latn-eng_Latn_chrf++,fuv_Latn-eng_Latn_ter
0,facebook/nllb-200-distilled-600M,flores,eng_Latn-fuv_Latn,1.74,2.44,16.95,137.93,10.27,12.42,30.72,93.29
1,facebook/nllb-200-distilled-600M,bouquet,eng_Latn-fuv_Latn,4.85,5.81,20.88,135.70,14.75,15.95,32.04,82.87


In [23]:
# Free up GPU memory
gc.collect()
torch.cuda.empty_cache()

## OPUS-MT

In [24]:
OPUS_MODEL_NAME = "Helsinki-NLP/opus-mt-tc-bible-big-mul-mul"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OPUS_PIPELINE = pipeline("translation", model=OPUS_MODEL_NAME, device=device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/991M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/292 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Device set to use cuda


In [25]:
def marian_inference(model_name, src_texts, src_lang, tgt_lang, **kwargs):
    # Prepend language tags to the source texts
    tgt_lang = tgt_lang.split('_')[0]
    src_texts = [f">>{tgt_lang}<< {text}" for text in src_texts]

    results = OPUS_PIPELINE(src_texts, **kwargs)

    return [res['translation_text'] for res in results]

In [26]:
opus_mt_results = evaluate_mt_model(OPUS_MODEL_NAME, marian_inference, datasets, batch_size=128)
opus_mt_results

Evaluating on flores dataset...
Translating from English to fuv_Latn...


Results for English to fuv_Latn: {'eng_Latn-fuv_Latn_bleu': 0.66, 'eng_Latn-fuv_Latn_spbleu': 0.77, 'eng_Latn-fuv_Latn_chrf++': 7.2, 'eng_Latn-fuv_Latn_ter': 113.56}
Translating from fuv_Latn to English...
Results for fuv_Latn to English: {'fuv_Latn-eng_Latn_bleu': 2.54, 'fuv_Latn-eng_Latn_spbleu': 3.34, 'fuv_Latn-eng_Latn_chrf++': 18.53, 'fuv_Latn-eng_Latn_ter': 151.79}

Evaluating on bouquet dataset...
Translating from English to fuc_Latn...
Results for English to fuc_Latn: {'eng_Latn-fuc_Latn_bleu': 0.98, 'eng_Latn-fuc_Latn_spbleu': 1.13, 'eng_Latn-fuc_Latn_chrf++': 9.92, 'eng_Latn-fuc_Latn_ter': 143.37}
Translating from fuc_Latn to English...
Results for fuc_Latn to English: {'fuc_Latn-eng_Latn_bleu': 1.09, 'fuc_Latn-eng_Latn_spbleu': 1.56, 'fuc_Latn-eng_Latn_chrf++': 15.49, 'fuc_Latn-eng_Latn_ter': 157.27}

Translating from English to fuv_Latn...
Results for English to fuv_Latn: {'eng_Latn-fuv_Latn_bleu': 0.88, 'eng_Latn-fuv_Latn_spbleu': 0.5, 'eng_Latn-fuv_Latn_chrf++': 5.48, 'en

Your input_length: 492 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Results for English to dag_Latn: {'eng_Latn-dag_Latn_bleu': 0.05, 'eng_Latn-dag_Latn_spbleu': 0.02, 'eng_Latn-dag_Latn_chrf++': 8.37, 'eng_Latn-dag_Latn_ter': 111.27}
Translating from dag_Latn to English...
Results for dag_Latn to English: {'dag_Latn-eng_Latn_bleu': 0.24, 'dag_Latn-eng_Latn_spbleu': 0.21, 'dag_Latn-eng_Latn_chrf++': 12.41, 'dag_Latn-eng_Latn_ter': 175.67}



,model,dataset,language_pair,eng_Latn-fuv_Latn_bleu,eng_Latn-fuv_Latn_spbleu,eng_Latn-fuv_Latn_chrf++,eng_Latn-fuv_Latn_ter,fuv_Latn-eng_Latn_bleu,fuv_Latn-eng_Latn_spbleu,fuv_Latn-eng_Latn_chrf++,...,fuc_Latn-eng_Latn_chrf++,fuc_Latn-eng_Latn_ter,eng_Latn-dag_Latn_bleu,eng_Latn-dag_Latn_spbleu,eng_Latn-dag_Latn_chrf++,eng_Latn-dag_Latn_ter,dag_Latn-eng_Latn_bleu,dag_Latn-eng_Latn_spbleu,dag_Latn-eng_Latn_chrf++,dag_Latn-eng_Latn_ter
0,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,flores,eng_Latn-fuv_Latn,0.66,0.77,7.20,113.56,2.54,3.34,18.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,bouquet,eng_Latn-fuc_Latn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.49,157.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,bouquet,eng_Latn-fuv_Latn,0.88,0.50,5.48,136.01,2.47,3.08,17.28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,udhr,eng_Latn-dag_Latn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.05,0.02,8.37,111.27,0.24,0.21,12.41,175.67


# Save Results

In [28]:
# Combine all results into a single DataFrame
all_results_df = pd.concat([nllb_3b_results, nllb_600m_results, opus_mt_results], ignore_index=True)
all_results_df.to_csv("MT_evaluation_results.csv", index=False)

all_results_df

,model,dataset,language_pair,eng_Latn-fuv_Latn_bleu,eng_Latn-fuv_Latn_spbleu,eng_Latn-fuv_Latn_chrf++,eng_Latn-fuv_Latn_ter,fuv_Latn-eng_Latn_bleu,fuv_Latn-eng_Latn_spbleu,fuv_Latn-eng_Latn_chrf++,...,fuc_Latn-eng_Latn_chrf++,fuc_Latn-eng_Latn_ter,eng_Latn-dag_Latn_bleu,eng_Latn-dag_Latn_spbleu,eng_Latn-dag_Latn_chrf++,eng_Latn-dag_Latn_ter,dag_Latn-eng_Latn_bleu,dag_Latn-eng_Latn_spbleu,dag_Latn-eng_Latn_chrf++,dag_Latn-eng_Latn_ter
0,facebook/nllb-200-3.3B,flores,eng_Latn-fuv_Latn,3.97,5.28,22.94,100.88,11.55,13.72,32.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,facebook/nllb-200-3.3B,bouquet,eng_Latn-fuv_Latn,6.50,9.63,24.51,94.96,18.08,19.04,36.27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,facebook/nllb-200-distilled-600M,flores,eng_Latn-fuv_Latn,1.74,2.44,16.95,137.93,10.27,12.42,30.72,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,facebook/nllb-200-distilled-600M,bouquet,eng_Latn-fuv_Latn,4.85,5.81,20.88,135.70,14.75,15.95,32.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,flores,eng_Latn-fuv_Latn,0.66,0.77,7.20,113.56,2.54,3.34,18.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,bouquet,eng_Latn-fuc_Latn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.49,157.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,bouquet,eng_Latn-fuv_Latn,0.88,0.50,5.48,136.01,2.47,3.08,17.28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Helsinki-NLP/opus-mt-tc-bible-big-mul-mul,udhr,eng_Latn-dag_Latn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.05,0.02,8.37,111.27,0.24,0.21,12.41,175.67
